In [0]:
from pyspark.sql.functions import to_date, sum as _sum, count as _count, avg as _avg
df_silver_check = spark.table("portfolio.raw_data.silver_taxi_trips")
df_gold = (df_silver_check
    .withColumn("trip_date", to_date("tpep_pickup_datetime"))
    .groupBy("trip_date")
    .agg(
        _count("*").alias("total_trips"),
        _sum("fare_amount").alias("total_revenue"),
        _avg("trip_distance").alias("avg_trip_distance"),
        _avg("fare_amount").alias("avg_fare")
    )
    .orderBy("trip_date")
)

(df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("portfolio.raw_data.gold_daily_taxi_summary"))

print("Gold table created!")

In [0]:
df_gold_check = spark.table("portfolio.raw_data.gold_daily_taxi_summary")
print("Gold row count:", df_gold_check.count())
df_gold_check.display()